In [13]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [19]:
#CONEXIONES DESTINO
DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [15]:
# MAESTROS GRANDES
paciente = pd.read_sql_query(f"SELECT id_paciente,per_sec FROM dim_paciente", con=connection2)
#personal = pd.read_sql_query(f"SELECT id_personal,num_doc FROM dim_personal", con=connection2)
paciente['per_sec'] = pd.to_numeric(paciente['per_sec'], errors='coerce').astype('Int64')
personal = pd.read_sql_query(f"SELECT id_personal,num_doc FROM dim_personal", con=connection2)

In [20]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_proceso=6", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_proceso=6", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]

In [21]:
dias_por_intervalo = 20

# Inicializa la fecha actual
fecha_actual = fecha_ini

while fecha_actual <= fecha_fin:
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	fecha_ini_intervalo = fecha_actual
	fecha_fin_intervalo = fecha_actual + timedelta(days=dias_por_intervalo - 1)

	if fecha_fin_intervalo > fecha_fin:
		fecha_fin_intervalo = fecha_fin

	fecha_ini_str = fecha_ini_intervalo.strftime('%Y-%m-%d')
	fecha_fin_str = (fecha_fin_intervalo + timedelta(days=1)).strftime('%Y-%m-%d')
	fecha_fin_display_str = fecha_fin_intervalo.strftime('%Y-%m-%d')

	print(f"Procesando de {fecha_ini_str} al {fecha_fin_display_str}")

	now1 = datetime.now()
	now2 = datetime.now()

	# Restar 20 días
	now2 = now2 - timedelta(days=20)

	# Convertir a formato 'YYYY-MM-DD'
	now2 = now2.strftime('%Y-%m-%d')

	query=f"UPDATE etl_act SET fec_act ='{now1}' WHERE id_proceso=6"

	c1= text(query)
	connection2.execute(c1)

	tabla='ctanm10'
	col_tabla = "ATENOMFEC"
	dat= "cext03_essi"
	col_dat='fec_ate'


	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()


	query1 = f"""
	SELECT
		c.ATENOMCENASICOD,
		c.ATENOMACTMEDNUM,
		c.ATENOMFEC,
		c.ATENOMHOR,
		c.ATENOMEVAL,
		c.ATENOMPLANTRA,
		c.ATENOMTRAT,
		c.ATENOMCSECOD,
		c.ATENOMCPSCOD,
		c.RESATENOMEDCOD,
		c.ATENOMESTREGCOD,
		c.ATENOMCREFEC,
		c.ATENOMMODFEC,
		c.ATENOMPROCENASICOD,
		c.ATENOMPROAREHOSCOD,
		c.ATENOMPROSERVHOSCOD,
		c.ATENOMPROACTCOD,
		c.ATENOMPROACTESPCOD,
		c.ATENOMPROPERASISDOCIDENNUM,
		c.ATENOMTURHORINI,
		c.ATENOMTURHORFIN,
		c.ATENOMNUMATECOD,
		c.ATENOMULTREGFEC,
		c.ATENOMPACSECNUM,
		d.ATENMDDIAGCOD,
		d.ATENMDCONDDIAGCOD,
		d.ATENMDDIAGORD,
		d.ATENMDTIPODIAGCOD,
		d.ATENMDCASODIAGCOD,
		d.ATENMDALTAFLG
	FROM {tabla} c
	LEFT OUTER JOIN CTDAN10 d		 
		ON c.ATENOMORICENASICOD = d.ATENOMORICENASICOD 
		AND c.ATENOMCENASICOD = d.ATENOMCENASICOD
		AND c.ATENOMACTMEDNUM = d.ATENOMACTMEDNUM
	WHERE c.{col_tabla} >= TO_DATE('{fecha_ini_str}', 'YYYY-MM-DD') AND c.{col_tabla} < TO_DATE('{fecha_fin_str}', 'YYYY-MM-DD')
	"""

	base1 = pd.read_sql_query(query1, con=connection0)
	connection0.close()
	engine0.dispose()

	base1 = base1.rename(columns={
		"atenomcenasicod": "cod_cas",
		"atenomactmednum": "act_med",
		"atenomfec": "fec_ate",
		"atenomhor": "hor_ate",
		"atenomeval": "atenom_val",
		"atenomplantra": "atenom_plantra",
		"atenomtrat": "atenom_trat",
		"atenomcsecod": "cod_cse",
		"atenomcpscod": "cod_cps",
		"resatenomedcod": "cod_resatennom",
		"atenomestregcod": "cod_est",
		"atenomcrefec": "fec_cre",
		"atenommodfec": "fec_mod",
		"atenomprocenasicod": "cod_caspro",
		"atenomproarehoscod": "cod_are",
		"atenomproservhoscod": "cod_ser",
		"atenomproactcod": "cod_act",
		"atenomproactespcod": "cod_sub",
		"atenomproperasisdocidennum": "num_doc",
		"atenomturhorini": "hor_ini",
		"atenomturhorfin": "hor_fin",
		"atenomnumatecod": "cod_ate",
		"atenomultregfec": "fec_ult_reg",
		"atenompacsecnum": "per_sec",
		"atenmddiagcod": "cod_diag",
		"atenmdconddiagcod": "cod_conddiag",
		"atenmddiagord": "diag_ord",
		"atenmdtipodiagcod": "cod_tipdiag",
		"atenmdcasodiagcod": "cod_casodiag",
		"atenmdaltaflg": "alta_flg"
	})

	base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 1", con=connection2)

	print(base1.shape)
	
	base1['per_sec'] = pd.to_numeric(base1['per_sec'], errors='coerce').astype('Int64')

	base1=pd.merge(base1,paciente,how='left',on='per_sec')	

	base1=base1.drop('per_sec',axis=1)

	print(base1.shape)

	personal['num_doc'] = personal['num_doc'].str.strip()
	base1['num_doc'] = base1['num_doc'].str.strip()
	base1=pd.merge(base1,personal,how='left',on='num_doc')

	base1=base1.drop('num_doc',axis=1)

	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	base1['cod_cas'] = base1['cod_cas'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas')

	base1=base1.drop('cod_cas',axis=1)

	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	cas=cas.rename(columns={"id_cas":"id_caspro"})
	cas=cas.rename(columns={"cod_cas":"cod_caspro"})
	base1['cod_caspro'] = base1['cod_caspro'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_caspro')

	csep = pd.read_sql_query(f"SELECT id_csep,cod_cse FROM dim_csep", con=connection2)
	csep['cod_cse'] = csep['cod_cse'].str.strip()
	base1['cod_cse'] = base1['cod_cse'].str.strip()
	base1=pd.merge(base1,csep,how='left',on='cod_cse')

	base1=base1.drop('cod_cse',axis=1)

	cps = pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
	cps['cod_cps'] = cps['cod_cps'].str.strip()
	base1['cod_cps'] = base1['cod_cps'].str.strip()
	base1=pd.merge(base1,cps,how='left',on='cod_cps')

	base1=base1.drop('cod_cps',axis=1)

	resatennom = pd.read_sql_query(f"SELECT id_resatennom,cod_resatennom FROM dim_resatennom", con=connection2)
	resatennom['cod_resatennom'] = resatennom['cod_resatennom'].str.strip()
	base1['cod_resatennom'] = base1['cod_resatennom'].str.strip()
	base1=pd.merge(base1,resatennom,how='left',on='cod_resatennom')

	base1=base1.drop('cod_resatennom',axis=1)

	estreg = pd.read_sql_query(f"SELECT id_estreg,cod_est FROM dim_estreg", con=connection2)
	estreg['cod_est'] = estreg['cod_est'].str.strip()
	base1['cod_est'] = base1['cod_est'].str.strip()
	base1=pd.merge(base1,estreg,how='left',on='cod_est')

	base1=base1.drop('cod_est',axis=1)

	are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
	are['cod_are'] = are['cod_are'].str.strip()
	base1['cod_are'] = base1['cod_are'].str.strip()
	base1=pd.merge(base1,are,how='left',on='cod_are')

	base1=base1.drop('cod_are',axis=1)

	serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
	serv['cod_ser'] = serv['cod_ser'].str.strip()
	base1['cod_ser'] = base1['cod_ser'].str.strip()
	base1=pd.merge(base1,serv,how='left',on='cod_ser')

	base1=base1.drop('cod_ser',axis=1)

	activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
	activi['cod_act'] = activi['cod_act'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	activi=activi.rename(columns={"id_activi":"id_acti"})
	base1=pd.merge(base1,activi,how='left',on='cod_act')

	subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
	subacti['cod_act'] = subacti['cod_act'].str.strip()
	subacti['cod_sub'] = subacti['cod_sub'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	base1['cod_sub'] = base1['cod_sub'].str.strip()
	subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
	subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
	base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
	base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
	subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
	base1 = pd.merge(base1,subacti,how='left',on='KEY')

	base1=base1.drop('KEY', axis=1)
	base1=base1.drop('cod_act',axis=1)
	base1=base1.drop('cod_sub',axis=1)

	diagcod = pd.read_sql_query(f"SELECT id_cie,cod_diag FROM dim_cie10", con=connection2)
	diagcod['cod_diag'] = diagcod['cod_diag'].str.strip()
	base1['cod_diag'] = base1['cod_diag'].str.strip()
	base1=pd.merge(base1,diagcod,how='left',on='cod_diag')

	base1=base1.drop('cod_diag',axis=1)

	conddiag = pd.read_sql_query(f"SELECT id_conddiag,cod_conddiag FROM dim_conddiag", con=connection2)
	conddiag['cod_conddiag'] = conddiag['cod_conddiag'].str.strip()
	base1['cod_conddiag'] = base1['cod_conddiag'].str.strip()
	base1=pd.merge(base1,conddiag,how='left',on='cod_conddiag')

	base1=base1.drop('cod_conddiag',axis=1)

	tipodiag = pd.read_sql_query(f"SELECT id_tipdiag,cod_tipdiag FROM dim_tipdiag", con=connection2)
	tipodiag['cod_tipdiag'] = tipodiag['cod_tipdiag'].str.strip()
	base1['cod_tipdiag'] = base1['cod_tipdiag'].str.strip()
	base1=pd.merge(base1,tipodiag,how='left',on='cod_tipdiag')

	base1=base1.drop('cod_tipdiag',axis=1)

	casodiag = pd.read_sql_query(f"SELECT id_casodiag,cod_casodiag FROM dim_casodiag", con=connection2)
	casodiag['cod_casodiag'] = casodiag['cod_casodiag'].str.strip()
	base1['cod_casodiag'] = base1['cod_casodiag'].str.strip()
	base1=pd.merge(base1,casodiag,how='left',on='cod_casodiag')

	base1=base1.drop('cod_casodiag',axis=1)

	print(base1.shape)

	df1_columns = set(base1.columns)
	df2_columns = set(base2.columns) 
	different_columns = df2_columns - df1_columns
	different_columns

	borrando = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha_ini_str}' and {col_dat} <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	comunes = set(base1.columns).intersection(set(base2.columns)) 
	base1 = base1[list(comunes)]

	#limpieza x00
	base1 = base1.applymap(lambda x: x.replace('\x00', '') if isinstance(x, str) else x)

	base1.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=500000)

	fecha_actual = fecha_fin_intervalo + timedelta(days=1)

	finproceso = time.time()
	tiempoproceso = finproceso - inicioTiempo
	tiempoproceso = round(tiempoproceso, 3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso) + " segundos")

query2=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_proceso=6"
c2= text(query2)
connection2.execute(c2)

Procesando de 2024-08-16 al 2024-09-04
(1398291, 30)
(1398295, 30)
(1415803, 31)
Proceso completado satisfactoriamente en 558.268 segundos
Procesando de 2024-09-05 al 2024-09-24
(6813, 30)
(6813, 30)
(6867, 31)
Proceso completado satisfactoriamente en 54.349 segundos
Procesando de 2024-09-25 al 2024-10-14
(1, 30)
(1, 30)
(1, 31)
Proceso completado satisfactoriamente en 9.791 segundos
Procesando de 2024-10-15 al 2024-11-03
(0, 30)
(0, 30)
(0, 31)
Proceso completado satisfactoriamente en 2.371 segundos
Procesando de 2024-11-04 al 2024-11-23
(1, 30)
(1, 30)
(1, 31)
Proceso completado satisfactoriamente en 10.278 segundos
Procesando de 2024-11-24 al 2024-12-13
(0, 30)
(0, 30)
(0, 31)
Proceso completado satisfactoriamente en 2.16 segundos
Procesando de 2024-12-14 al 2025-01-01
(1, 30)
(1, 30)
(1, 31)
Proceso completado satisfactoriamente en 10.688 segundos


In [18]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()